# 통계·확률 프로젝트 × 데이터 분석기법 더하기 × 정직한 차트 과제

- **근거 1:** 「데이터 분석 기법 더하기」 PDF 10기법 — 기존 로또 과제와 연결
- **근거 2:** 「빅데이터와 저널리즘 · 기사용 차트 문법」 — 날짜 병합·어긋난 날·6규칙·수정전후 3쌍
- **근거 3:** 「데이터 시각화 한눈에 보기」 — 비교/변화/관계/분포, 산점도·히트맵·검수

**자료 분리:** 뉴스·PM10은 가상으로 생성한 예제이고, 로또 949~1248회는 사용자가 제공한 과거 실제 번호입니다. 실제 뉴스나 대기질 관련 보도를 만들어 낸 것이 아닙니다.

## 1. 질문 → 자료 찾기 → 정제 → 분석 → 검증 → 기사

In [1]:
from pathlib import Path
import pandas as pd, numpy as np, json
from IPython.display import display, Image
BASE=Path.cwd()
news=pd.read_csv(BASE/'data/synthetic_news_articles.csv')
air=pd.read_csv(BASE/'data/synthetic_pm10_daily.csv')
print('가상 기사',len(news),'건, 가상 PM10',len(air),'일')
display(news.head(2));display(air.head(2))

가상 기사 425 건, 가상 PM10 85 일


## 2. 날짜별 기사 수와 PM10 왼쪽 병합 (`how="left"`)

In [2]:
daily=news.groupby('day').size().reset_index(name='news_count')
merged=air.merge(daily,on='day',how='left',validate='one_to_one')
merged['news_count']=merged['news_count'].fillna(0).astype(int)
assert len(merged)==len(air)==85 and merged.news_count.sum()==len(news)
print('병합 85행, 기사 없는 날짜',sum(merged.news_count==0))
display(merged.head())

병합 85행, 기사 없는 날짜 4


## 3. 상관관계와 어긋난 날 찾기

PM10 중앙값보다 낮은데 기사 수가 상위 10%를 초과한 날짜를 골라봅니다. **상관은 인과가 아닙니다.**

In [3]:
corr=merged.news_count.corr(merged.pm10_ug_m3)
low=merged.pm10_ug_m3<merged.pm10_ug_m3.median()
many=merged.news_count>merged.news_count.quantile(.9)
discordant=merged.loc[low & many].sort_values('news_count',ascending=False)
print('Pearson r',round(corr,6),'어긋난 날짜',len(discordant))
display(discordant)

Pearson r -0.164901 어긋난 날짜 4


## 4. IQR로 PM10 높은데 기사가 적은 날짜 찾기

In [4]:
q1,q3=merged.pm10_ug_m3.quantile([.25,.75])
threshold=q3+1.5*(q3-q1)
quiet=merged[(merged.pm10_ug_m3>threshold)&(merged.news_count<merged.news_count.median())]
print('IQR 상단',threshold,'날짜 수',len(quiet))
display(quiet)

IQR 상단 75.0 날짜 수 2


## 5. 실제 과거 로또 60회 번호 빈도와 시간순 검증

로또는 별개의 실제 과거 자료로, 가상 미세먼지/기사 수와 혼합하지 않습니다.

In [5]:
lotto=pd.read_csv(BASE/'data/lotto_949_1248_sample.csv')
assert len(lotto)==300
recent=lotto.sort_values('drwNo').tail(60)
frequency=pd.Series(recent[[f'n{i}' for i in range(1,7)]].to_numpy().ravel()).value_counts()
print('최근60회 최다 번호 빈도');display(frequency.head(8))
result=json.loads((BASE/'data/analysis_results.json').read_text(encoding='utf-8'))
print('백테스트 과거 150회',result['backtest_means'])

최근60회 최다 번호 빈도


백테스트 과거 150회 {'최근 60회 빈도': 0.7733, '지수가중 빈도': 0.7067, '무작위 기준선': 0.9133}


## 6. 정직한 차트 6규칙 — 3쌍 수정 전·후

**수정 전** 이미지는 축 절단·이중축·다색 강조 등 **의도적인 잘못된 표현**입니다. 실제 보도에는 수정 후 이미지만 사용합니다.

### 1. 막대는 0부터

In [6]:
display(Image(filename='charts/01_before_truncated_axis.png',width=550))
display(Image(filename='charts/01_after_zero_axis.png',width=550))

### 2. 서로 다른 단위는 두 칸

In [7]:
display(Image(filename='charts/02_before_dual_axis.png',width=550))
display(Image(filename='charts/02_after_shared_dates.png',width=550))

### 3. 강조색 하나·라벨

In [8]:
display(Image(filename='charts/03_before_unfocused.png',width=550))
display(Image(filename='charts/03_after_one_highlight.png',width=550))

## 7. 관계와 분포 — 산점도·히트맵

In [9]:
display(Image(filename='charts/04_scatter_relationship.png',width=650))
display(Image(filename='charts/05_heatmap_lotto.png',width=650))

## 8. 재계산 검증

모든 CSV·PNG·SVG는 `python run_analysis.py`로 재생성할 수 있습니다. 로또 표본은 1248회까지이며 미래번호 예측 성능을 보장하지 않습니다.

In [10]:
assert abs(corr-result['synthetic_corr_news_pm10'])<1e-6
assert len(discordant)==len(result['synthetic_lowpm_highnews'])
assert len(recent)==60
print('데이터 검수 완료: CSV 수치·예제 날짜·전/후 차트 3쌍·추가 차트 2장')

데이터 검수 완료: CSV 수치·예제 날짜·전/후 차트 3쌍·추가 차트 2장
